# 36 — Grid and anchor-based object detection (YOLO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/ai-dl/36-yolo/lab.ipynb) [![Open in LinguAgent](https://img.shields.io/badge/Open%20in-LinguAgent-1f6f5c)](https://linguagent.com/courses/ai-dl/yolo?utm_source=github&utm_medium=labs)

*~55 minutes · 20 tests*

Module 27 gives you the primitives — IoU, NMS. This module is the mechanics
specific to a one-shot, grid-based detector: turning ground-truth boxes into
a training target a convolutional grid can predict, decoding that grid back
into boxes, matching a box to the anchor shape it resembles, the loss
weighting that keeps a mostly-empty grid from collapsing to "predict
nothing everywhere," and the greedy matching step that turns raw detections
into the (confidence, correct) pairs module 32's `average_precision`
actually needs.

Work in `exercises.py`.

**How this notebook works:** run the first cell; then, section by section, read, write the code in the cell under it, and run the check below it. Stuck? `lab.solution("name")` shows the reference — try first, it's worth it.

In [ ]:
#@title ▶️ Run me first — fetches this lab and its checker
import os, sys, subprocess
HOME = "/content/linguagent-labs"
if "google.colab" in sys.modules:
    if not os.path.isdir(HOME):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/mrshahabian/linguagent-labs.git", HOME], check=True)
else:  # Jupyter, VS Code or a Codespace: this notebook already sits inside the repo
    HOME = os.path.abspath(os.path.join(os.getcwd(), *[".."] * 3)) if os.path.isdir("../../../labkit") else os.getcwd()
sys.path.insert(0, HOME)
import labkit
lab = labkit.open("ai-dl/yolo")

In [ ]:
# What the exercises below use — run this cell.
import torch

## What you'll build

| Name | You'll implement | Shows up in real code as |
|---|---|---|
| `encode_grid_targets` | the function | Turning ground-truth boxes into a trainable grid target |
| `decode_grid_predictions` | the function | Turning a model's raw grid output back into boxes |
| `best_anchor_for_box` | the function | YOLOv2/v3's shape-based anchor assignment |
| `yolo_loss_components` | the function | The loss-weighting that fixes YOLO's class imbalance |
| `match_detections_to_ground_truth` | the function | Feeding real detections into module 32's `average_precision` |

---

## 1. One cell is responsible for each box

```python
col = min(int(center_x / cell_size), grid_size - 1)
row = min(int(center_y / cell_size), grid_size - 1)
target[row, col, 0] = 1.0                 # objectness
target[row, col, 1:5] = x_offset, y_offset, width, height
target[row, col, 5 + class_id] = 1.0
```

YOLO's central move: divide the image into a `grid_size x grid_size`
grid, and whichever cell contains a box's **center** — not every cell it
overlaps — is the one cell responsible for predicting it. That cell's
slot holds `[objectness, x, y, w, h, one-hot class]`, where `x, y` are the
center's offset **within the cell** (0 to 1), and `w, h` are the box's
size as a fraction of the whole image. Every other cell stays all zero,
which doubles as exactly the right training signal for "no object here."

```python
>>> target = encode_grid_targets(boxes, class_ids, grid_size=7, img_size=280, num_classes=5)
>>> target[..., 0].sum()          # one cell marked responsible per box
tensor(2.)
```

**In `exercises.py`:** `encode_grid_targets`.

In [ ]:
def encode_grid_targets(boxes, class_ids, grid_size, img_size, num_classes):
    """Encode ground-truth boxes into a (grid_size, grid_size, 5+num_classes) tensor.

    YOLO's central idea: divide the image into a grid_size x grid_size
    grid: whichever cell contains a box's CENTER is the one cell
    "responsible" for predicting it - not every cell that overlaps it.
    That cell's slot becomes [objectness=1, x, y, w, h, one-hot class],
    where x, y are the center's offset WITHIN the cell (0 to 1, not
    absolute pixels) and w, h are the box's width/height as a fraction of
    the whole image. Cells with no assigned box are left at all zero,
    which is exactly what makes objectness=0 the right "no object here"
    signal for the loss to train against.

    boxes: (N, 4) in (x1, y1, x2, y2) absolute pixel coordinates.
    class_ids: (N,) integer class index per box.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§1", globals())

## 2. Decoding: only confident cells produce a detection

```python
confidence = cell[0].item()
if confidence <= conf_threshold:
    continue
center_x = (col + x_offset) * cell_size
...
class_id = int(torch.argmax(cell[5:]))
```

The reverse of encoding: for every cell above `conf_threshold`, convert
the relative `(x, y, w, h)` back to absolute pixel coordinates and read
off the highest-scoring class. Most cells in a real image have no
object, so most cells never produce a detection at all — that sparsity
is exactly what the loss in §4 has to be weighted against, or training
collapses to predicting "no object" everywhere, which is *usually*
right and therefore a tempting local minimum.

```python
>>> detections = decode_grid_predictions(target, grid_size=7, img_size=280, conf_threshold=0.5)
>>> detections[0]
(tensor([10., 10., 50., 90.]), 1.0, 1)
```

**In `exercises.py`:** `decode_grid_predictions`.

In [ ]:
def decode_grid_predictions(grid_output, grid_size, img_size, conf_threshold):
    """Reverse encode_grid_targets: grid tensor -> list of (box, confidence, class).

    Only cells with objectness above conf_threshold produce a detection -
    most of a grid is "no object here" for any real image, which is
    exactly the class imbalance yolo_loss_components has to account for.
    Converts each surviving cell's relative (x, y, w, h) back to absolute
    (x1, y1, x2, y2) pixel coordinates, and reports the class with the
    highest score in that cell.

    Returns a list of (box_tensor, confidence, class_id) tuples, one per
    detection, in row-major cell order.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§2", globals())

## 3. Anchor assignment: match by shape, not position

```python
inter_w = min(box_w, anchor_w)
inter_h = min(box_h, anchor_h)
iou = intersection / (box_area + anchor_area - intersection)
```

YOLOv2/v3 predict boxes relative to a handful of predefined **anchor**
shapes (typical width/height pairs, usually chosen by clustering the
training set's own boxes) instead of predicting raw width and height
directly. Assignment picks whichever anchor's *shape* the ground-truth
box resembles most — computed as if both were centered at the same
point, so only their width and height matter, never where either
actually sits in the image. This decouples "what shape is this object"
from "where is it": an anchor tuned for tall, narrow boxes doesn't
compete unfairly against one tuned for wide, short boxes just because of
where either happens to be positioned in a particular training image.

```python
>>> anchors = [(10, 10), (50, 20), (20, 60)]
>>> best_anchor_for_box((40, 16), anchors)   # comparable scale + ratio to anchor 1
1
```

**In `exercises.py`:** `best_anchor_for_box`.

In [ ]:
def best_anchor_for_box(box_wh, anchor_whs):
    """Pick the anchor whose SHAPE best matches box_wh, ignoring position.

    YOLOv2/v3 assign a ground-truth box to whichever of several predefined
    anchor shapes (typical width/height ratios, chosen up front by
    clustering the training set's boxes) it resembles most - not by IoU at
    the box's actual location, but by IoU as if both the box and every
    anchor were centered at the same point. This decouples "what shape is
    this object" from "where is it", so an anchor tuned for tall, narrow
    boxes (a person) doesn't compete unfairly with one tuned for wide,
    short boxes (a car) just because of where either happens to sit in the
    image.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§3", globals())

## 4. The loss weighting that keeps training from collapsing

```python
coord_loss = lambda_coord * sum((pred[obj][:, 1:5] - target[obj][:, 1:5]) ** 2)
obj_loss = sum((pred[obj][:, 0] - target[obj][:, 0]) ** 2)
noobj_loss = lambda_noobj * sum((pred[noobj][:, 0] - target[noobj][:, 0]) ** 2)
class_loss = sum((pred[obj][:, 5:] - target[obj][:, 5:]) ** 2)
```

A grid is mostly "no object" cells — for any realistic image, empty
cells vastly outnumber cells with a real box in them. A plain
sum-of-squared-errors loss would be dominated by driving every empty
cell's confidence to exactly zero, drowning out the rarer, more
important signal from cells that *do* contain something. Two
hyperparameters fix this in opposite directions: `lambda_coord > 1`
amplifies the box-coordinate loss (getting a real object's box right
matters more than raw squared error alone would weight it), and
`lambda_noobj < 1` shrinks the confidence loss for empty cells, so their
sheer numerical majority doesn't dominate the gradient.

```python
>>> losses = yolo_loss_components(pred, target, lambda_coord=5.0, lambda_noobj=0.5)
>>> losses.keys()
dict_keys(['coord', 'obj', 'noobj', 'class', 'total'])
```

**In `exercises.py`:** `yolo_loss_components`.

In [ ]:
def yolo_loss_components(pred, target, lambda_coord=5.0, lambda_noobj=0.5):
    """The three weighted pieces of the YOLO loss, summed.

    A grid is mostly empty - most cells have no object in them - so a
    naive sum-of-squared-errors loss would be dominated by driving every
    empty cell's confidence to exactly zero, drowning out the (rarer, more
    important) signal from cells that DO contain an object. Two
    hyperparameters fix the imbalance in opposite directions:

      lambda_coord (>1): amplify the box-coordinate loss, since getting a
      real object's box right matters more than the raw squared-error
      scale would suggest.

      lambda_noobj (<1): shrink the confidence loss for cells with no
      object, so their sheer numerical majority doesn't dominate training.

    Both masks come from target's own objectness channel - only cells
    where target[..., 0] == 1 contribute coordinate and class loss; every
    cell contributes objectness loss, split into the responsible and
    not-responsible masks.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§4", globals())

## 5. From detections to `average_precision`'s inputs

```python
ordered = sorted(detections, key=lambda d: d[1], reverse=True)
for box, confidence, _class_id in ordered:
    # match against the best UNCLAIMED ground-truth box above iou_threshold
```

Module 32's `average_precision` takes `(confidences, labels)` — it
doesn't know what a bounding box is. Getting from raw detections to
those inputs is a real step of its own: sort by confidence, then greedily
match each detection to the best still-unclaimed ground-truth box above
an IoU threshold. **Each ground-truth box can be claimed once.** A second
detection overlapping an already-claimed box is a false positive even at
high IoU — in a working detector that duplicate should have been
suppressed by NMS before evaluation ever runs, so counting it as correct
here would hide a real problem.

```python
>>> confidences, labels = match_detections_to_ground_truth(detections, gt_boxes, iou_threshold=0.5)
>>> average_precision(confidences, labels)   # from module 32
```

**In `exercises.py`:** `match_detections_to_ground_truth`.

In [ ]:
def match_detections_to_ground_truth(detections, ground_truth_boxes, iou_threshold=0.5):
    """Greedily match predictions to ground truth for a mAP-style evaluation.

    Detections are matched HIGHEST CONFIDENCE FIRST, and each ground-truth
    box can be claimed by at most one detection - a second detection that
    also overlaps an already-claimed box is a false positive, even at
    high IoU, because in a real detector it would be a duplicate the
    model should have suppressed itself (this is why NMS runs before
    evaluation, not just before display). This is the step module 32's
    average_precision doesn't include on its own: turning raw
    (box, confidence) detections into the (confidence, is_correct) pairs
    average_precision actually consumes.

    detections: list of (box, confidence, class_id), any order.
    Returns (confidences, labels) - labels[i] is 1 if detections[i] (after
    sorting by confidence) was matched to an unclaimed ground-truth box
    above iou_threshold, else 0.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§5", globals())

---

## The exercises

| Function | What it teaches |
|---|---|
| `encode_grid_targets` | One responsible cell per box; relative coordinates |
| `decode_grid_predictions` | Confidence thresholding; most cells predict nothing |
| `best_anchor_for_box` | Shape-only IoU for anchor assignment |
| `yolo_loss_components` | `lambda_coord`/`lambda_noobj` against class imbalance |
| `match_detections_to_ground_truth` | Greedy IoU matching feeding a real mAP computation |

Every function in this table has a stub in `exercises.py` and test
coverage in `test_exercises.py` — a clean 1:1 mapping.

In [ ]:
def _iou_xyxy(box_a, box_b):
    """Single-pair IoU, (x1, y1, x2, y2) format - the same formula as
    module 27's box_iou, not a batched (N, M) version, since greedy
    matching below only ever needs one prediction against one ground
    truth at a time."""
    raise NotImplementedError("your code here")

## Check your understanding

1. **Why does only the cell containing a box's *center* get marked
   responsible, instead of every cell the box overlaps?**
   <details><summary>Answer</summary>If every overlapping cell were
   responsible, multiple cells would predict the same object, and the
   model would need some other mechanism to avoid multiple redundant
   detections for one box. Assigning exactly one cell (by center) makes
   the training target unambiguous — each object has exactly one cell
   whose prediction it is.</details>

2. **Why does anchor assignment compute IoU as if the box and every
   anchor were centered at the same point, instead of using their real
   positions?**
   <details><summary>Answer</summary>Anchor assignment is answering "what
   shape does this box look like," not "where is it" — the grid cell
   already encodes position. Comparing real positions would make the
   match depend on where the object happens to sit in a particular
   image, which has nothing to do with which anchor shape best fits
   it.</details>

3. **What would happen to training if `lambda_noobj` were 1.0 instead of
   a small fraction?**
   <details><summary>Answer</summary>Empty cells vastly outnumber cells
   with a real object, so the objectness loss from empty cells would
   dominate the total gradient. The model would be pushed hard toward
   "predict no object everywhere," a solution that's usually correct and
   therefore an easy local minimum — genuine objects would get a
   comparatively weak training signal.</details>

4. **Why does `match_detections_to_ground_truth` process detections in
   confidence order and let each ground-truth box be claimed only
   once?**
   <details><summary>Answer</summary>Confidence order means the model's
   most-trusted predictions get first claim on a ground-truth box, which
   matches how a real system would present or act on its results. Only
   allowing one claim per ground-truth box makes a duplicate detection
   of the same object count as a false positive — exactly the failure
   mode NMS exists to prevent before evaluation, so evaluation should
   still catch it if NMS didn't.</details>

5. **`best_anchor_for_box`'s IoU calculation ignores the box's aspect
   ratio if the box is much smaller than every anchor. What's the
   underlying reason?**
   <details><summary>Answer</summary>The overlap is capped by
   <code>min(box_w, anchor_w)</code> and <code>min(box_h, anchor_h)</code>
   — for a box much smaller than every anchor, that intersection is close
   to the box's own area regardless of which anchor it's compared
   against, so the union (and therefore the IoU) ends up dominated by
   each anchor's own size rather than by how well the shapes actually
   match. In practice this means the metric favors similarly-SIZED
   anchors, not just similarly-shaped ones, when the box is
   small.</details>

## Interview questions this module answers

- *Walk me through how YOLO turns a set of ground-truth boxes into
  something a grid of predictions can be trained against.*
- *Why does YOLO need anchor boxes, and how is a box assigned to one?*
- *This detector's loss collapses to predicting no objects — why, and
  how do you fix it?*
- *How do you turn a set of raw detections into a precision-recall curve
  or an mAP number?*

## Finished? Check everything, then record it

The first cell checks the whole unit. The second records your result in LinguAgent (needs your lab token from [https://linguagent.com/settings#labs](https://linguagent.com/settings#labs) — in Colab, add it once as a secret named `LINGUAGENT_TOKEN` under 🔑 in the left bar).

In [ ]:
lab.check_all(globals())

In [ ]:
lab.sync()

**Next:** [DL 14 - Classical image processing](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/ai-dl/37-image-processing/lab.ipynb) · [the lesson in LinguAgent](https://linguagent.com/courses/ai-dl/yolo)